# Predicing the precursor charge with macine learning

In [5]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

In [6]:
import re

AA_BASIC = set("KRH")
AA_ACIDIC = set("DE")

def extract_features(row):
    seq = row["peptide_sequence"]

    # remove modifications
    clean_seq = re.sub(r"\[.*?\]", "", seq)

    intensities = np.asarray(row["intensities_raw"])

    log_int = np.log1p(intensities)

    ions = row["matched_ions"]

    b_nums = []
    y_nums = []

    for ion in ions:
        if ion.startswith("b"):
            b_nums.append(int(ion[1:]))
        elif ion.startswith("y"):
            y_nums.append(int(ion[1:]))

    return {
        "length": len(clean_seq),
        "n_KRH": sum(aa in AA_BASIC for aa in clean_seq),
        "n_DE": sum(aa in AA_ACIDIC for aa in clean_seq),

        "n_b": len(b_nums),
        "n_y": len(y_nums),
        "max_b": max(b_nums) if b_nums else 0,
        "max_y": max(y_nums) if y_nums else 0,

        "int_mean": log_int.mean(),
        "int_std": log_int.std(),
        "int_max": log_int.max(),
        "int_median": np.median(log_int),
        "int_sum": log_int.sum(),
    }

In [10]:
df = pd.read_parquet("data_for_student.parquet")
meta = pd.read_parquet("metadata_for_student.parquet")
merged = df.merge(meta, on=["raw_file", "scan_number"], how="inner")


known = merged[merged.precursor_charge.notna()].copy()

X = pd.DataFrame(
    known.apply(extract_features, axis=1).tolist()
)

y = known["precursor_charge"].astype(int)

In [11]:
classes = np.sort(y.unique())

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weight = dict(zip(classes, weights))

In [14]:
clf = HistGradientBoostingClassifier(
    max_depth=8,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           1       0.02      0.18      0.04        11
           2       0.99      0.99      0.99     42766
           3       0.98      0.98      0.98     25035
           4       0.92      0.82      0.87      2189
           5       0.14      0.39      0.20        44
           6       0.00      0.00      0.00         1

    accuracy                           0.98     70046
   macro avg       0.51      0.56      0.51     70046
weighted avg       0.99      0.98      0.98     70046



### Cat Boost because gradient boosting and nya nya

do 
```python
$ uv pip install catboost
```
to install required libraries

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    depth=8,
    iterations=500,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    verbose=100
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)

print(classification_report(y_test, pred))

0:	learn: 1.6935669	total: 252ms	remaining: 2m 5s
100:	learn: 0.2734873	total: 24.5s	remaining: 1m 36s
200:	learn: 0.1372662	total: 50.8s	remaining: 1m 15s
300:	learn: 0.0863693	total: 1m 18s	remaining: 51.7s
400:	learn: 0.0598670	total: 1m 43s	remaining: 25.5s
499:	learn: 0.0466130	total: 2m 6s	remaining: 0us
              precision    recall  f1-score   support

           1       0.07      0.82      0.13        11
           2       1.00      0.99      0.99     42766
           3       0.99      0.97      0.98     25035
           4       0.75      0.96      0.85      2189
           5       0.48      0.70      0.57        44
           6       0.00      0.00      0.00         1

    accuracy                           0.98     70046
   macro avg       0.55      0.74      0.59     70046
weighted avg       0.99      0.98      0.98     70046

